In [1]:
from structures.card import Carta, Pokémon, Trainer, Energy
from structures.deck import Deck
from structures.expansion import Expansion
from structures.similarity import Similarity

from pathlib import Path
import json, traceback
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm 

from __future__ import annotations
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Iterable, Tuple, List
from itertools import chain
from tqdm import tqdm

DECKS_PATH = Path('data/tournament_decks')


Definiamo un paio di funzioni per caricare tutti i file in una directory oppure solo una lista ristretta. 
da questi file viene estratto un numero variabile di mazzi (Deck). 
Se sono presenti errori, il mazzo viene scartato

In [2]:
def process_file(file_path):
	"""
	Carica un singolo file JSON e ne estrae i deck.
	Ritorna (decks, errore): decks è lista di Deck; errore è dict o None.
	"""
	try:
		with file_path.open('r', encoding='utf-8') as f:
			data = json.load(f)
		decks = Deck.from_tournament_data(data)
		return decks, None
	except Exception as e:
		return [], {
			"file": str(file_path),
			"errore": str(e),
			"traceback": traceback.format_exc()
		}

def _normalize_paths(paths: Path | Iterable[Path], pattern: str, recursive: bool) -> List[Path]:
	# Accetta: directory, file, lista/iterabile
	if isinstance(paths, Path):
		if paths.is_dir():
			return sorted(paths.rglob(pattern) if recursive else paths.glob(pattern))
		else:
			return [paths] if paths.suffix.lower() == ".json" else []
	# Iterabile di Path
	ps = list(paths)
	return [p for p in ps if p.is_file() and p.suffix.lower() == ".json"]

def load_decks(
		paths: Path | Iterable[Path],
		pattern: str = "*.json",
		recursive: bool = False,
		max_workers: int = 10,
		strict: bool = False
		# count: int = 0
	) -> Tuple[List["Deck"], List[dict]]:
	"""
	Carica deck da più file JSON in parallelo.

	- paths: directory, file singolo o iterabile di Path
	- pattern: glob (usato se 'paths' è una directory)
	- recursive: usa rglob sul pattern
	- max_workers: thread pool size
	- strict: se True, solleva al primo errore (per pipeline CI)
	"""
	file_list = _normalize_paths(paths, pattern, recursive)
	if not file_list:
		return [], []

	workers = max(1, min(max_workers, len(file_list)))
	chunksize = max(1, len(file_list) // (workers * 4))

	decks: List["Deck"] = []
	errori: List[dict] = []

	iter_results = None
	with ThreadPoolExecutor(max_workers=workers) as executor:
		iter_results = executor.map(process_file, file_list, chunksize=chunksize)

		iter_results = tqdm(iter_results, total=len(file_list), unit="file",
		                    desc="Caricamento JSON")

		for result_decks, err in iter_results:
			for deck in result_decks:
				if deck.len() > 0:
					decks.append(deck)
			if err:
				if strict:
					# Mostra il contesto e interrompe
					raise RuntimeError(f"Errore su {err.get('file')}: {err.get('errore')}")
				errori.append(err)
			# if count != 0 and len(decks) >= count:
			# 	break
			

	return decks, errori

Carichiamo tramite threading tutti i mazzi presenti in DECKSPATH
Questo processo è lungo e tedioso, per fortuna va fatto solo una volta

# Esempio di come i mazzi sono estrapolati -> salvati -> caricati
#### Estrapola

In [ ]:
# Decommenta queste linee sotto per estrapolare solo i primi n file
# n=1
# DECKS_PATH = [p for p in (DECKS_PATH.glob("*.json"))][:n]
decks, errori = load_decks(DECKS_PATH, max_workers=10)
print(f"Decks caricati: {len(decks)}")

Caricamento JSON:  70%|███████   | 71/101 [18:19<07:44, 15.49s/file]


Nessun set trovato con ptcgo 'SV1S'
Gardevoir ex non trovato in SV1S, scarto il mazzo
Nessun set trovato con ptcgo 'SV5A'
Iron Thorns ex non trovato in SV5a, scarto il mazzo
Nessun set trovato con ptcgo 'S9A'
Ralts non trovato in S9a, scarto il mazzo
Nessun set trovato con ptcgo 'SV1S'
Gardevoir ex non trovato in SV1S, scarto il mazzo
Nessun set trovato con ptcgo 'SV3'
Cleffa non trovato in SV3, scarto il mazzo
'Carta sv4a-139 non trovata nel set sv4a.json'
Pidgeot ex non trovato in SV4a, scarto il mazzo
Nessun set trovato con ptcgo 'SV6'
Munkidori non trovato in SV6, scarto il mazzo
Nessun set trovato con ptcgo 'S10D'
Origin Forme Dialga V non trovato in S10D, scarto il mazzo
Nessun set trovato con ptcgo 'S10B'
Snorlax non trovato in S10b, scarto il mazzo
Nessun set trovato con ptcgo 'SV6'
Dreepy non trovato in SV6, scarto il mazzo
Nessun set trovato con ptcgo 'SV6'
Dreepy non trovato in SV6, scarto il mazzo
Nessun set trovato con ptcgo 'S12A'
Origin Forme Dialga V non trovato in S12a

RuntimeError: Errore su data/tournament_decks/0000129.json: 'NoneType' object has no attribute 'get'

In [ ]:
if errori:
    print(f"{len(errori)} file con errori:")
    for e in errori:
        print(f" - {e['file']}: {e['errore']}")

#### Salva

In [ ]:
OUTPUT_PATH = Path("cache/decks.json")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump([deck.to_dict() for deck in decks], f, ensure_ascii=False, indent=2)



#### Carica

In [ ]:
# with OUTPUT_PATH.open("r", encoding="utf-8") as f:
#     data = json.load(f)
# mazzi = [Deck.from_dict(d) for d in data]

# for deck in mazzi:
#     print(deck.carte)